# Task 1: Chunking Strategy, Index Selection, and Reranker Comparison

Part A compares sentence vs fixed-length chunking. 

Part B compares index strategies. HNSW and LSH could not be evaluated on the full corpus due to RAM constraints and are discussed theoretically. 

Part C compares reranking methods on the full corpus.

In [2]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import time, pickle, gc, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from eval_utils import (
    load_questions, build_chunks, embed_chunks,
    evaluate_retrieval, retrieval_metrics,
    precompute_query_embeddings, print_summary
)
from textwave.modules.extraction.embedding import Embedding
from textwave.modules.retrieval.index.bruteforce import BruteForceIndex
from textwave.modules.retrieval.reranker import Reranker

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

os.makedirs('figures', exist_ok=True)
print('Setup complete!')

Setup complete!


In [3]:
questions = load_questions(deduplicate=True)
print(f'Questions: {len(questions)}')
print(questions['DifficultyFromAnswerer'].value_counts())

Questions: 628
DifficultyFromAnswerer
easy        214
medium      174
hard         95
too hard      9
too easy      4
Name: count, dtype: int64


## Part A: Chunking Strategy Comparison

Compare sentence chunking (3 sentences, overlap 1) vs fixed-length chunking (500 chars, overlap 50) using BruteForce index at k=5.

In [4]:
embedder = Embedding('all-MiniLM-L6-v2')
K = 5
chunking_results = {}

for strategy, params in [
    ('sentence', dict(num_sentences=3, overlap_size=1)),
    ('fixed-length', dict(chunk_size=500, chunk_overlap=50)),
]:
    print(f'\nBuilding {strategy} chunks...')
    chunks, source_files = build_chunks(strategy=strategy, **params)
    print(f'  Total chunks: {len(chunks)}')

    print('  Embedding...')
    embeddings = embed_chunks(chunks)
    dim = embeddings.shape[1]

    idx = BruteForceIndex(dim)
    idx.add(embeddings, chunks)

    print('  Evaluating...')
    results = evaluate_retrieval(questions, idx, source_files, embedder, k=K)
    chunking_results[strategy] = results

    p = results['precision@k'].mean()
    r = results['recall@k'].mean()
    f = results['f1@k'].mean()
    print(f'  P@5: {p:.4f} | R@5: {r:.4f} | F1@5: {f:.4f}')

    if strategy == 'sentence':
        np.save('sentence_embeddings.npy', embeddings)
        with open('sentence_chunks.pkl', 'wb') as f_out:
            pickle.dump((chunks, source_files), f_out)
        print('  Saved to disk.')

    del idx, embeddings, chunks, source_files
    gc.collect()

print('\nDone!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6286.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Building sentence chunks...
  Total chunks: 17161
  Embedding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12666.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Evaluating...
  P@5: 0.6885 | R@5: 0.8869 | F1@5: 0.7416
  Saved to disk.

Building fixed-length chunks...
  Total chunks: 9664
  Embedding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5212.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Evaluating...
  P@5: 0.6510 | R@5: 0.8503 | F1@5: 0.7037

Done!


In [5]:
# Precompute query embeddings
query_vecs = precompute_query_embeddings(questions, save_path='query_embeddings.npy')
del embedder
gc.collect()
print('Done!')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5593.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved 628 query embeddings to query_embeddings.npy!
Done!


In [6]:
rows = []
for strategy, df in chunking_results.items():
    rows.append({
        'Strategy': strategy,
        'Precision@5': round(df['precision@k'].mean(), 4),
        'Recall@5': round(df['recall@k'].mean(), 4),
        'F1@5': round(df['f1@k'].mean(), 4),
    })
chunking_summary = pd.DataFrame(rows)
print('Table 1: Chunking Strategy Comparison')
print(chunking_summary.to_string(index=False))
chunking_summary.to_csv('results_chunking.csv', index=False)

Table 1: Chunking Strategy Comparison
    Strategy  Precision@5  Recall@5   F1@5
    sentence       0.6885    0.8869 0.7416
fixed-length       0.6510    0.8503 0.7037


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, ['Precision@5', 'Recall@5', 'F1@5']):
    ax.bar(chunking_summary['Strategy'], chunking_summary[metric], color=['steelblue', 'coral'])
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
plt.suptitle('Figure 1: Chunking Strategy Comparison (BruteForce, k=5)')
plt.tight_layout()
plt.savefig('figures/chunking_comparison.png', dpi=150)
plt.close()
print('Figure 1 saved!')

Figure 1 saved!


## Part B: Index Strategy Comparison

BruteForce is evaluated on the full corpus. 

HNSW and LSH could not be run due to RAM constraints! HNSW builds a multi-layer graph on top of the raw vectors, and even on a 1,000-chunk subset the kernel crashed. Their properties are discussed in the case analysis based on established benchmarks.

In [8]:
# Index comparison summary
# BruteForce empirical, HNSW/LSH theoretical
index_summary = pd.DataFrame([
    {
        'Index': 'BruteForce',
        'Recall@5': round(chunking_results['sentence']['recall@k'].mean(), 4),
        'Precision@5': round(chunking_results['sentence']['precision@k'].mean(), 4),
        'F1@5': round(chunking_results['sentence']['f1@k'].mean(), 4),
        'Build Time': 'fast',
        'Query Time': 'O(n)',
        'Note': 'exact, full corpus',
    },
    {
        'Index': 'HNSW',
        'Recall@5': 'N/A',
        'Precision@5': 'N/A',
        'F1@5': 'N/A',
        'Build Time': 'slow',
        'Query Time': 'O(log n)',
        'Note': 'RAM exceeded on hardware',
    },
    {
        'Index': 'LSH',
        'Recall@5': 'N/A',
        'Precision@5': 'N/A',
        'F1@5': 'N/A',
        'Build Time': 'fast',
        'Query Time': 'O(1) approx',
        'Note': 'RAM exceeded on hardware',
    },
])
print('Table 2: Index Strategy Comparison')
print(index_summary.to_string(index=False))
index_summary.to_csv('results_index.csv', index=False)

Table 2: Index Strategy Comparison
     Index Recall@5 Precision@5    F1@5 Build Time  Query Time                     Note
BruteForce   0.8869      0.6885  0.7416       fast        O(n)       exact, full corpus
      HNSW      N/A         N/A     N/A       slow    O(log n) RAM exceeded on hardware
       LSH      N/A         N/A     N/A       fast O(1) approx RAM exceeded on hardware


## Part C: Reranker Comparison

In [9]:
# Reload from disk
query_vecs = np.load('query_embeddings.npy')
full_embeddings = np.load('sentence_embeddings.npy')
with open('sentence_chunks.pkl', 'rb') as f:
    full_chunks, full_sources = pickle.load(f)

dim = full_embeddings.shape[1]
bf_idx = BruteForceIndex(dim)
bf_idx.add(full_embeddings, full_chunks)
print(f'BruteForce index ready: {bf_idx.ntotal} chunks!')

RETRIEVE_K = 20
RERANK_K = 5
reranker_results = {}

# No-reranker baseline
reranker_results['none'] = chunking_results['sentence']
print(f"Baseline Recall@5: {reranker_results['none']['recall@k'].mean():.4f}")

BruteForce index ready: 17161 chunks!
Baseline Recall@5: 0.8869


In [10]:
for rtype in ['tfidf', 'bow', 'cross_encoder', 'sequential']:
    print(f'\nEvaluating {rtype}...')
    try:
        reranker = Reranker(type=rtype)
        rows = []
        for i, (_, row) in enumerate(questions.iterrows()):
            query = row['Question']
            query_vec = query_vecs[i]
            cand_chunks, cand_indices, _ = bf_idx.search_with_indices(query_vec, RETRIEVE_K)
            cand_sources = [full_sources[j] for j in cand_indices if j < len(full_sources)]

            if rtype == 'sequential':
                ranked_docs, ranked_idx, _ = reranker.rerank(
                    query, cand_chunks, seq_k1=10, seq_k2=RERANK_K
                )
            else:
                ranked_docs, ranked_idx, _ = reranker.rerank(query, cand_chunks)

            reranked_sources = [cand_sources[j] for j in ranked_idx[:RERANK_K] if j < len(cand_sources)]
            m = retrieval_metrics(reranked_sources, row['ArticleFile'], RERANK_K)
            m['difficulty'] = row['DifficultyFromAnswerer']
            rows.append(m)

        reranker_results[rtype] = pd.DataFrame(rows)
        recall = reranker_results[rtype]['recall@k'].mean()
        print(f'  Recall@5: {recall:.4f}')
    except Exception as e:
        print(f'  Failed: {e}')


Evaluating tfidf...


Loading weights: 100%|██████████| 41/41 [00:00<00:00, 11139.17it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Recall@5: 0.8981

Evaluating bow...


Loading weights: 100%|██████████| 41/41 [00:00<00:00, 14340.10it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Recall@5: 0.8917

Evaluating cross_encoder...


Loading weights: 100%|██████████| 41/41 [00:00<00:00, 16438.82it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Recall@5: 0.8997

Evaluating sequential...


Loading weights: 100%|██████████| 41/41 [00:00<00:00, 13289.53it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Recall@5: 0.8981


In [11]:
rows = []
for name, df in reranker_results.items():
    rows.append({
        'Reranker': name,
        'Precision@5': round(df['precision@k'].mean(), 4),
        'Recall@5': round(df['recall@k'].mean(), 4),
        'F1@5': round(df['f1@k'].mean(), 4),
    })
reranker_summary = pd.DataFrame(rows)
print('Table 3: Reranker Comparison')
print(reranker_summary.to_string(index=False))
reranker_summary.to_csv('results_reranker.csv', index=False)

Table 3: Reranker Comparison
     Reranker  Precision@5  Recall@5   F1@5
         none       0.6885    0.8869 0.7416
        tfidf       0.6567    0.8981 0.7252
          bow       0.6898    0.8917 0.7451
cross_encoder       0.6911    0.8997 0.7493
   sequential       0.6860    0.8981 0.7452


In [12]:
plt.figure(figsize=(10, 4))
plt.bar(reranker_summary['Reranker'], reranker_summary['Recall@5'], color='steelblue')
plt.title('Figure 3: Recall@5 by Reranking Strategy (full corpus)')
plt.ylabel('Recall@5')
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('figures/reranker_comparison.png', dpi=150)
plt.close()
print('Figure 3 saved!')

Figure 3 saved!
